# PostgreSQL connection with Bitwarden

Use a Python environment with DataBay installed. The Spark runtime must provide
the PostgreSQL JDBC driver and Delta Lake support for this example.

The Bitwarden CLI must be installed and logged in. Create a login item containing
Username, Password, and custom fields `host`, `database`, and optionally `port`.
Set `bitwarden_item_id` below to the item's ID. The database host must be
reachable from the Spark runtime.

Run the cells in order. Each JDBC operation requests the Bitwarden master
password using hidden input, retrieves the connection settings, and locks
the vault before connecting to PostgreSQL.


In [ ]:
from databay.etl import Octopus, bitwarden_connection
from databay.runtime import spark_direct

# The Spark infrastructure must provide the PostgreSQL JDBC driver.
spark = spark_direct()
spark.sparkContext.setLogLevel("ERROR")


In [ ]:
bitwarden_item_id = "YOUR_BITWARDEN_ITEM_ID"

octopus = Octopus(
    spark=spark,
    engine="postgresql",
    connection_provider=lambda: bitwarden_connection(bitwarden_item_id),
)

frames = octopus.read_jdbc([("SELECT 1 AS connection_ok", "connection_check")])
frames["connection_check"].show()


## Seed PostgreSQL

Create three synthetic datasets: 100 customers, 20 products and 1,000 orders.
Write them into PostgreSQL using `write_jdbc`. A unique prefix separates each
run's tables. The database account needs CREATE TABLE permission in `public`.


In [ ]:
from uuid import uuid4
from pyspark.sql import functions as F

run_id = uuid4().hex[:12]
prefix = f"databay_demo_{run_id}"
target_schema = f"databay_demo_{run_id}"

customers = spark.range(1, 101).select(
    F.col("id").alias("customer_id"),
    F.concat(F.lit("Customer "), F.col("id")).alias("customer_name"),
    F.concat(F.lit("customer"), F.col("id"), F.lit("@example.invalid")).alias("email"),
)
products = spark.range(1, 21).select(
    F.col("id").alias("product_id"),
    F.concat(F.lit("Product "), F.col("id")).alias("product_name"),
    (F.col("id") * F.lit(2.5)).cast("decimal(10,2)").alias("unit_price"),
)
orders = spark.range(1, 1001).select(
    F.col("id").alias("order_id"),
    ((F.col("id") - 1) % 100 + 1).alias("customer_id"),
    ((F.col("id") - 1) % 20 + 1).alias("product_id"),
    ((F.col("id") - 1) % 5 + 1).cast("int").alias("quantity"),
)

demo_frames = {
    f"{prefix}_customers": customers,
    f"{prefix}_products": products,
    f"{prefix}_orders": orders,
}
expected_counts = dict(zip(demo_frames, (100, 20, 1000)))
print("PostgreSQL source tables:", *demo_frames, sep="\n")


In [ ]:
# Writes only these generated tables. Existing tables are never overwritten.
octopus.write_jdbc(
    data=demo_frames,
    target_schema="public",
    mode="errorifexists",
    num_partitions=2,
    batch_size=1000,
    connect_timeout=10,
    query_timeout=60,
    socket_timeout=120,
)
print(f"Seeded {len(demo_frames)} PostgreSQL tables (100 / 20 / 1,000 rows).")


## Download PostgreSQL tables into Delta

`feed_spark` reads the three source tables and saves them sequentially into
a new Spark namespace. A single progress line shows the current table and its
position in the batch. The Spark runtime supplies the warehouse and Delta support.


In [ ]:
queries = [
    (f'SELECT * FROM public."{table_name}"', table_name)
    for table_name in demo_frames
]

octopus.feed_spark(
    queries=queries,
    target_schema=target_schema,
    table_format="delta",
    batch_size=1000,
    connect_timeout=10,
    query_timeout=60,
    socket_timeout=120,
)


In [ ]:
# Count the persisted Delta tables, then preview the downloaded orders.
for table_name, expected in expected_counts.items():
    target = f"{target_schema}.{table_name}"
    actual = spark.table(target).count()
    assert actual == expected, f"{target}: expected {expected} rows, got {actual}"
    print(f"{target}: {actual:,} rows OK")

spark.table(f"{target_schema}.{prefix}_orders").orderBy("order_id").show(10)


In [ ]:
# Run when finished using the returned DataFrames and this Spark session.
del frames, octopus
spark.stop()
